<a href="https://colab.research.google.com/github/Myria255/BOOTCAMP-TTA/blob/main/W6D4_LoRA_ExerciseXP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Parameter-Efficient Fine-Tuning — LoRA Implementation Exercises

**Developers Institute & Sira Labs — Week 6, Day 4**

Ce notebook complète les six exercices demandés :

1. Implémentation de `LoRALayer`
2. Création de `LinearWithLoRA`
3. Application de LoRA à une couche linéaire
4. Fusion des matrices LoRA avec le poids original
5. Création d'un MLP et remplacement de ses couches linéaires
6. Gel des poids originaux et entraînement des seuls paramètres LoRA



In [ ]:

# Imports et configuration générale
import copy
import time
import torch
import torch.nn as nn
import torch.nn.functional as F

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

RANDOM_SEED = 123
torch.manual_seed(RANDOM_SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("PyTorch version :", torch.__version__)
print("Device utilisé  :", DEVICE)


## Exercice 1 — Implémenter `LoRALayer`

Pour une entrée $x$, LoRA apprend deux matrices de faible rang :

* $A \in \mathbb{R}^{d_{\text{in}} \times r}$
* $B \in \mathbb{R}^{r \times d_{\text{out}}}$

La transformation produite par la couche LoRA est :

$$
\Delta y = \alpha , xAB
$$

La matrice $B$ est initialisée avec des zéros. Par conséquent, au début de l’entraînement, la contribution de LoRA est nulle et la sortie du modèle reste identique à celle du modèle original.


In [ ]:

class LoRALayer(nn.Module):
    """Low-Rank Adaptation layer.

    The layer computes: alpha * (x @ A @ B)
    """

    def __init__(self, in_dim: int, out_dim: int, rank: int, alpha: float):
        super().__init__()

        if rank <= 0:
            raise ValueError("rank doit être strictement positif.")

        std_dev = 1 / torch.sqrt(torch.tensor(rank, dtype=torch.float32))

        self.A = nn.Parameter(torch.randn(in_dim, rank) * std_dev)
        self.B = nn.Parameter(torch.zeros(rank, out_dim))
        self.alpha = alpha

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.alpha * (x @ self.A @ self.B)


# Test de la couche LoRA
torch.manual_seed(RANDOM_SEED)

test_lora = LoRALayer(in_dim=4, out_dim=3, rank=2, alpha=1.0)
test_input = torch.randn(2, 4)
test_output = test_lora(test_input)

print("Forme de l'entrée :", test_input.shape)
print("Forme de la sortie:", test_output.shape)
print("Sortie initiale :\n", test_output)
print("La sortie est nulle au départ :", torch.allclose(
    test_output, torch.zeros_like(test_output)
))



## Exercices 2 et 3 — `LinearWithLoRA` et vérification de l'initialisation

`LinearWithLoRA` additionne :

1. la sortie de la couche linéaire originale ;
2. l'adaptation de faible rang produite par LoRA.

Comme `B` commence à zéro, la sortie doit être identique à celle de la couche
linéaire originale juste après l'initialisation.


In [ ]:

class LinearWithLoRA(nn.Module):
    """Wrap an existing nn.Linear layer with a LoRA adaptation."""

    def __init__(self, linear: nn.Linear, rank: int, alpha: float):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            in_dim=linear.in_features,
            out_dim=linear.out_features,
            rank=rank,
            alpha=alpha,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear(x) + self.lora(x)


# Couche linéaire et tenseur de test
torch.manual_seed(RANDOM_SEED)

layer = nn.Linear(in_features=4, out_features=3)
x = torch.randn(2, 4)

original_output = layer(x)

# La même couche originale est enveloppée par LoRA
layer_lora_1 = LinearWithLoRA(layer, rank=2, alpha=1.0)
lora_output = layer_lora_1(x)

print("Entrée :\n", x)
print("\nCouche originale :\n", layer)
print("\nSortie originale :\n", original_output)
print("\nSortie LinearWithLoRA :\n", lora_output)
print("\nSorties identiques au départ :", torch.allclose(
    original_output, lora_output, atol=1e-7
))


## Exercice 4 — Fusionner LoRA avec le poids original

La couche linéaire de PyTorch effectue le calcul suivant :

$$
y = xW^T + b
$$

La matrice LoRA $AB$ possède la dimension `(in_features, out_features)`.

Cependant, `linear.weight` possède la dimension `(out_features, in_features)`.

Pour fusionner la matrice LoRA avec le poids original, il faut donc transposer $AB$ :

$$
W_{\text{combiné}} = W + \alpha (AB)^T
$$

Cette fusion permet d’intégrer directement l’adaptation LoRA dans le poids de la couche linéaire.


In [ ]:

class LinearWithLoRAMerged(nn.Module):
    """Linear layer whose original and LoRA weights are combined at forward time."""

    def __init__(self, linear: nn.Linear, rank: int, alpha: float):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            in_dim=linear.in_features,
            out_dim=linear.out_features,
            rank=rank,
            alpha=alpha,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        lora_update = self.lora.A @ self.lora.B
        combined_weight = (
            self.linear.weight
            + self.lora.alpha * lora_update.T
        )
        return F.linear(x, combined_weight, self.linear.bias)


# Créer une version fusionnée qui partage les mêmes valeurs que layer_lora_1
layer_lora_2 = LinearWithLoRAMerged(
    copy.deepcopy(layer_lora_1.linear),
    rank=2,
    alpha=1.0,
)
layer_lora_2.lora.load_state_dict(layer_lora_1.lora.state_dict())

output_separate = layer_lora_1(x)
output_merged = layer_lora_2(x)

print("Sortie Linear + LoRA séparés :\n", output_separate)
print("\nSortie avec poids fusionné :\n", output_merged)
print("\nLes deux approches sont équivalentes :", torch.allclose(
    output_separate, output_merged, atol=1e-7
))



## Exercice 5 — MLP à trois couches

Le modèle reçoit une image MNIST aplatie de `28 × 28 = 784` caractéristiques et
produit dix logits, un pour chaque chiffre de 0 à 9.


In [ ]:

class MultilayerPerceptron(nn.Module):
    def __init__(
        self,
        num_features: int,
        num_hidden_1: int,
        num_hidden_2: int,
        num_classes: int,
    ):
        super().__init__()

        self.layers = nn.Sequential(
            nn.Linear(num_features, num_hidden_1),
            nn.ReLU(),
            nn.Linear(num_hidden_1, num_hidden_2),
            nn.ReLU(),
            nn.Linear(num_hidden_2, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # MNIST arrive sous la forme [batch, 1, 28, 28]
        x = torch.flatten(x, start_dim=1)
        return self.layers(x)


# Architecture
num_features = 28 * 28
num_hidden_1 = 128
num_hidden_2 = 64
num_classes = 10

# Paramètres d'entraînement
learning_rate = 0.001
num_epochs_pretrain = 3
num_epochs_lora = 2
BATCH_SIZE = 64

model = MultilayerPerceptron(
    num_features=num_features,
    num_hidden_1=num_hidden_1,
    num_hidden_2=num_hidden_2,
    num_classes=num_classes,
).to(DEVICE)

optimizer_pretrained = torch.optim.Adam(
    model.parameters(),
    lr=learning_rate,
)

print(model)
print("\nOptimiseur :", optimizer_pretrained)


## Chargement du dataset MNIST

In [ ]:

transform = transforms.ToTensor()

train_dataset = datasets.MNIST(
    root="data",
    train=True,
    transform=transform,
    download=True,
)

test_dataset = datasets.MNIST(
    root="data",
    train=False,
    transform=transform,
    download=True,
)

train_loader = DataLoader(
    dataset=train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)

test_loader = DataLoader(
    dataset=test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available(),
)

images, labels = next(iter(train_loader))

print("Dimensions du batch d'images :", images.shape)
print("Dimensions des labels         :", labels.shape)
print("Nombre d'images d'entraînement:", len(train_dataset))
print("Nombre d'images de test        :", len(test_dataset))


## Fonctions d'évaluation et d'entraînement

In [ ]:

def compute_accuracy(
    model: nn.Module,
    data_loader: DataLoader,
    device: torch.device,
) -> float:
    model.eval()
    correct_pred = 0
    num_examples = 0

    with torch.no_grad():
        for features, targets in data_loader:
            features = features.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            logits = model(features)
            predicted_labels = torch.argmax(logits, dim=1)

            num_examples += targets.size(0)
            correct_pred += (
                predicted_labels == targets
            ).sum().item()

    return 100.0 * correct_pred / num_examples


def train(
    num_epochs: int,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    train_loader: DataLoader,
    device: torch.device,
) -> list[float]:
    start_time = time.time()
    epoch_losses = []

    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0

        for batch_idx, (features, targets) in enumerate(train_loader):
            features = features.to(device, non_blocking=True)
            targets = targets.to(device, non_blocking=True)

            logits = model(features)
            loss = F.cross_entropy(logits, targets)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

            if batch_idx % 400 == 0:
                print(
                    f"Epoch {epoch + 1:02d}/{num_epochs:02d} | "
                    f"Batch {batch_idx:04d}/{len(train_loader):04d} | "
                    f"Loss {loss.item():.4f}"
                )

        average_loss = running_loss / len(train_loader)
        epoch_losses.append(average_loss)

        train_accuracy = compute_accuracy(model, train_loader, device)
        print(
            f"Epoch {epoch + 1:02d}/{num_epochs:02d} | "
            f"Loss moyenne: {average_loss:.4f} | "
            f"Accuracy train: {train_accuracy:.2f}%"
        )
        print(
            f"Temps écoulé: {(time.time() - start_time) / 60:.2f} min\n"
        )

    print(
        f"Temps total d'entraînement: "
        f"{(time.time() - start_time) / 60:.2f} min"
    )
    return epoch_losses


## Pré-entraînement du MLP original

In [ ]:

pretrain_losses = train(
    num_epochs=num_epochs_pretrain,
    model=model,
    optimizer=optimizer_pretrained,
    train_loader=train_loader,
    device=DEVICE,
)

original_test_accuracy = compute_accuracy(
    model,
    test_loader,
    DEVICE,
)

print(f"Accuracy test du modèle original : {original_test_accuracy:.2f}%")



## Remplacement des couches linéaires par LoRA

On copie le modèle pré-entraîné puis on remplace chacune de ses trois couches
`nn.Linear` par `LinearWithLoRAMerged`.

Comme les matrices `B` sont initialisées à zéro, le modèle LoRA doit donner les
mêmes prédictions que le modèle original avant le fine-tuning.


In [ ]:

model_lora = copy.deepcopy(model)

model_lora.layers[0] = LinearWithLoRAMerged(
    model_lora.layers[0],
    rank=4,
    alpha=8,
)
model_lora.layers[2] = LinearWithLoRAMerged(
    model_lora.layers[2],
    rank=4,
    alpha=8,
)
model_lora.layers[4] = LinearWithLoRAMerged(
    model_lora.layers[4],
    rank=4,
    alpha=8,
)

model_lora = model_lora.to(DEVICE)

accuracy_original_before = compute_accuracy(model, test_loader, DEVICE)
accuracy_lora_before = compute_accuracy(model_lora, test_loader, DEVICE)

print(model_lora)
print(f"\nAccuracy modèle original : {accuracy_original_before:.2f}%")
print(f"Accuracy modèle LoRA avant fine-tuning : {accuracy_lora_before:.2f}%")
print(
    "Même accuracy au départ :",
    abs(accuracy_original_before - accuracy_lora_before) < 1e-9,
)



## Exercice 6 — Geler les couches originales et entraîner uniquement LoRA

La fonction récursive ci-dessous désactive les gradients de toutes les couches
`nn.Linear`. Les matrices `A` et `B` des modules `LoRALayer` restent
entraînables.


In [ ]:

def freeze_linear_layers(module: nn.Module) -> None:
    """Freeze every original nn.Linear layer recursively."""
    for child in module.children():
        if isinstance(child, nn.Linear):
            for parameter in child.parameters():
                parameter.requires_grad = False
        else:
            freeze_linear_layers(child)


freeze_linear_layers(model_lora)

print("État des paramètres :")
for name, parameter in model_lora.named_parameters():
    print(f"{name:45s} | trainable = {parameter.requires_grad}")

trainable_parameters = [
    parameter
    for parameter in model_lora.parameters()
    if parameter.requires_grad
]

total_params = sum(p.numel() for p in model_lora.parameters())
trainable_params = sum(p.numel() for p in trainable_parameters)

print(f"\nParamètres totaux      : {total_params:,}")
print(f"Paramètres entraînables : {trainable_params:,}")
print(
    f"Part entraînable        : "
    f"{100 * trainable_params / total_params:.2f}%"
)

assert trainable_parameters, "Aucun paramètre LoRA entraînable détecté."
assert all(
    (
        (".lora.A" in name or ".lora.B" in name)
        if parameter.requires_grad
        else True
    )
    for name, parameter in model_lora.named_parameters()
), "Un paramètre autre que LoRA est encore entraînable."


In [ ]:

optimizer_lora = torch.optim.Adam(
    trainable_parameters,
    lr=learning_rate,
)

lora_losses = train(
    num_epochs=num_epochs_lora,
    model=model_lora,
    optimizer=optimizer_lora,
    train_loader=train_loader,
    device=DEVICE,
)

lora_test_accuracy = compute_accuracy(
    model_lora,
    test_loader,
    DEVICE,
)

print("\nRésultats finaux")
print("-" * 45)
print(f"Accuracy modèle original      : {original_test_accuracy:.2f}%")
print(f"Accuracy LoRA avant adaptation: {accuracy_lora_before:.2f}%")
print(f"Accuracy LoRA après adaptation: {lora_test_accuracy:.2f}%")



## Conclusion

- `LoRALayer` ajoute une mise à jour de faible rang sans modifier directement
  les poids pré-entraînés.
- L'initialisation de `B` à zéro garantit que le comportement initial du modèle
  reste inchangé.
- `LinearWithLoRA` et `LinearWithLoRAMerged` produisent des sorties
  équivalentes.
- Après le gel des couches originales, seules les matrices LoRA `A` et `B`
  sont optimisées.
- LoRA réduit fortement le nombre de paramètres entraînables, ce qui rend le
  fine-tuning plus léger en mémoire et en calcul.

### Soumission

Dans Colab :

1. `Fichier → Importer un notebook`
2. Sélectionner ce fichier `.ipynb`
3. `Fichier → Enregistrer une copie dans Drive`
4. `Partager → Accès général → Tous les utilisateurs disposant du lien`
5. Copier le lien public sur la plateforme DI
